In [0]:
from pyspark.sql.functions import *
from pyspark.sql import Window

In [0]:
# Load predictions from forecast table
predictions_table = spark.table(
    "agentdb.forecasting.forecast_predictions"
)

# Load training data to get target_sales_qty
training_data = spark.table(
    "agentdb.forecasting.forecast_training_dataset"
)

# Get the most recent target_sales_qty for each product/store
# This avoids cartesian product explosion (1912x per product/store)
latest_actuals = (
    training_data
    .withColumn(
        "row_num",
        row_number().over(
            Window.partitionBy("product_key", "store_key")
            .orderBy(col("calendar_key").desc())
        )
    )
    .filter(col("row_num") == 1)
    .select("product_key", "store_key", "target_sales_qty")
)

# Join predictions with the most recent actuals only
predictions = (
    predictions_table
    .join(
        latest_actuals,
        on=["product_key", "store_key"],
        how="inner"
    )
)

In [0]:
# Calculate evaluation metrics
evaluation = (

    predictions

    .groupBy(
        "model_name"
    )

    .agg(

        avg(
            abs(
                col("target_sales_qty")
                -
                col("predicted_demand")
            )
        ).alias("mae"),

        sqrt(
            avg(
                pow(
                    col("target_sales_qty")
                    -
                    col("predicted_demand"),
                    2
                )
            )
        ).alias("rmse"),

        avg(

            when(
                col("target_sales_qty") != 0,

                abs(
                    (
                        col("target_sales_qty")
                        -
                        col("predicted_demand")
                    )
                    /
                    col("target_sales_qty")
                )
            )

        ).alias("mape"),

        (
            sum(
                abs(
                    col("target_sales_qty")
                    -
                    col("predicted_demand")
                )
            )
            /
            sum(
                abs(
                    col("target_sales_qty")
                )
            )
        ).alias("wmape")
    )
)

# Add metadata columns
evaluation = (
    evaluation
    .withColumn(
        "evaluation_date",
        current_date()
    )
    .withColumn(
        "created_at",
        current_timestamp()
    )
)

# Write to table
evaluation.write \
    .mode("overwrite") \
    .saveAsTable(
        "agentdb.forecasting.forecast_metrics"
    )

In [0]:
product_accuracy = (

    predictions

    .groupBy(
        "product_key"
    )

    .agg(
        avg(
            abs(
                col("target_sales_qty")
                -
                col("predicted_demand")
            )
        ).alias("product_mae")
    )
)

display(product_accuracy)

In [0]:
store_accuracy = (

    predictions

    .groupBy(
        "store_key"
    )

    .agg(
        avg(
            abs(
                col("target_sales_qty")
                -
                col("predicted_demand")
            )
        ).alias("store_mae")
    )
)

display(store_accuracy)

In [0]:
forecast_model_comparison = (
    spark.table("agentdb.forecasting.forecast_metrics")
    .select(
        col("model_name").alias("Model"),
        col("wmape").alias("WMAPE")
    )
    .orderBy("WMAPE")
)

display(forecast_model_comparison)

In [0]:
winner = (
    evaluation
    .orderBy(col("wmape").asc())
    .limit(1)
)

winner = winner.withColumn("is_active", lit(True)).withColumn("selected_at", current_timestamp())

winner.select(
    "model_name", "is_active", "mae", "rmse", "mape", "wmape", "selected_at"
).write.mode("overwrite").saveAsTable("agentdb.forecasting.forecast_model_registry")

In [0]:
# Write individual prediction records with actual vs predicted values
predictions.withColumn(
    "evaluation_date",
    current_date()
).withColumn(
    "mae",
    abs(col("target_sales_qty") - col("predicted_demand"))
).withColumn(
    "rmse",
    pow(col("target_sales_qty") - col("predicted_demand"), 2)
).withColumn(
    "mape",
    when(
        col("target_sales_qty") != 0,
        abs((col("target_sales_qty") - col("predicted_demand")) / col("target_sales_qty"))
    ).otherwise(None)
).withColumn(
    "wmape",
    when(
        col("target_sales_qty") != 0,
        abs(col("target_sales_qty") - col("predicted_demand")) / abs(col("target_sales_qty"))
    ).otherwise(None)
).select(
    "evaluation_date",
    "product_key",
    "store_key",
    "forecast_horizon_days",
    col("target_sales_qty").cast("double").alias("actual_value"),
    col("predicted_demand").alias("predicted_value"),
    "mae",
    "rmse",
    "mape",
    "wmape",
    "created_at"
).write.mode("overwrite").saveAsTable("agentdb.evaluation.eval_forecast_accuracy")

print("✓ Wrote prediction-level evaluation records to eval_forecast_accuracy")
display(spark.table("agentdb.evaluation.eval_forecast_accuracy").limit(10))